# Indicador ODS-14.1.1a - Eutrofização Costeiro-Marinha do Brasil


## Ciclo de Vida dos Dados com CRISP-DM

Neste exercício vamos aplicar as etapas do **CRISP-DM** a criação deste indicador.

Objetivo geral:

> Usar como dados de entrada imagens do sensor SGLI (Second-Generation Global Imager) do satélite GCOM-C (Shikisai) da faz parte da missão GCOM (Global Change Observation Mission) desenvolvida pela JAXA (Agência de Exploração Aeroespacial do Japão)

As etapas segundo padrão CRISP-DM serão:

1. Compreensão do Negócio
2. Compreensão dos Dados
3. Preparação dos Dados
4. Modelagem
5. Avaliação
6. Implantação


# 1. Compreensão do Negócio

O IBGE precisa nacionalizar os indicadores ODS. Dentre os indicadores temos o ODS-14.1.1a. Este indicador é Índice de Eutrofização Costeira (Index of Coastal Eutrophication — ICEP). Ele é um subindicador oficial da Meta 14.1 dos Objetivos de Desenvolvimento Sustentável da ONU (ODS 14: Vida na Água).


## Pergunta do projeto

É possível calcular com sensoriamento remoto o indicador 14.1.1a com sensoriamento remoto internacional correspondentes a aferições locais ?

## Objetivo

1. Obter o dataset de Clorofila-a do site FTP da Agência Espacial Japonesa JAXA
2. Obter os polígonos vetoriais da Cadastro Naacional das Unidades de Conservação - CNUC - do site do MMA.
3. Transformar dados matatriciais em imagens GeoTiff;
4. Corrigir a projeção das imagens para mesma projeção das feições dos polígonos do CNUC;
5. Fazer o recorte das imagens para a área delimitada das Unidades de Conservação Costeiro-Marinhas;
4. construir um `DataFrame` multimodal;
5. O canal de cor da imagem em escala logarítmica;
6. Ajustar o modelo linear da imagem $f(y) = Pixel(w) + b $ , onde $f(x)$ é a massa seca de plancton, $Pixel$ é o valor da imagem, e $w$ (peso) e $b$ (vício) são valores de calibração do sensor.
7. Gerar, para cada imagem localizada no tempo e espaço (UC), um label (valor de placton dissolvido) em $\frac{mg}{m^3}$
8. Gerar a série rotulada.
6. separar dados de treino e teste;
7. treinar um modelo de regressão linear, devido a natureza do fenômeno;
8. Checar se modelo suporta sazonalidade, acrescentar mais variáveis de entrada;
8. avaliar o modelo (Teste) e calcular o erro. dados In situ vs dados Previstos;
9. usar o modelo para classificar uma nova observação.


# 2. Compreensão dos Dados

A entrada $x_1$ são as imagens da agencia japonesa. O moisaico será o CRHA global.

- Formato - (Matricial - NetCDF);
- Canais de Dados - 1 canal de cor de 16 bits;
- Resolução Temporal: 2018 a 2026
- Resolução Espacial: Global - Projeção eixo "x" de 0 a 360 - Necessita conversão.
- Variáveis de sensor: Estão embarcadas na matriz da imagem.

A entrada $x_2$ são feições vetoriais poliginares do Banco de dados CNUC (Cadastro Nacional de Unidades de Conservação) do M.M.A.

- Formato - SHP;
- Canais de Dados - 1 vetorial (geom) + atributos (características) tabulares de cada feição;
- Resolução Temporal: 1984 a 2026
- Resolução Espacial: EPSG:4674 - Sirgas 2000 geográfico
- variáveis anexas: (Conferir metados no site do CNUC).
- Usa geocódigo do IBGE ? SIM



## 2.1 Importar bibliotecas


Instalar o pacote tqdm para baixar as imagens da agência espacial Japonesa.

In [ ]:
!pip install tqdm

## 2.2 Executar o programa para obter as imagens matriciais multi-dimensionais netcdf

### 2.2.1 - Tipo de imagem com a qual vamos trabalhar

Aqui está a comparação didática entre os níveis de processamento de dados de satélite (**L1, L2 e L3**) em formato Markdown:

| Nível | Estágio do Dado | O que é / O que foi feito? | Formato / Geometria | Para quem é recomendado? |
| --- | --- | --- | --- | --- |
| **L1** *(Level-1)* | **Radiância Bruta** | Dados brutos medidos pelo sensor convertidos em valores físicos de radiância/reflectância no topo da atmosfera (TOA). | Organizado no formato da varredura da órbita (*swath* / "pista" percorrida pelo satélite). | Pesquisadores que desenvolvem novos algoritmos de calibração ou correção atmosférica. |
| **L2** *(Level-2)* | **Variável Física Instantânea** | Aplicação de correções atmosféricas para extrair a variável real na superfície (ex: concentração de clorofila-a em $mg/m^3$). | Ainda na geometria original da órbita/varredura (pixels com formatos e ângulos levemente distorcidos). | Estudos de dinâmica rápida que exigem o dado instantâneo de uma hora/dia específica. |
| **L3** *(Level-3)* | **Produto Mapeado e Agregado** | Dados do L2 **reprojetados em uma grade geográfica regular** (ex: *Plate Carrée*) e **agregados no tempo** (médias de 8 dias, mensais, etc.). | Matriz regular perfeita de Latitude e Longitude (grade de pixels uniformes). | **Análise ambiental e séries temporais** em Python/GIS. Elimina nuvens e dispensa pré-processamento. |

---

#### 2.2.1.1 Resumo do Fluxo de Dados

$$\text{L1 (Radiância no Satélite)} \xrightarrow[\text{Atmosférica}]{\text{Correção}} \text{L2 (Clorofila Instantânea)} \xrightarrow[\text{e Agregação Temporal}]{\text{Reprojeção em Grade}} \text{L3 (Matriz Final de 8 dias)}$$

#### 2.2.2.2 Projeção Utilizada nas Imagens: Geográfica Global (*Plate Carrée*)?

A projeção **Plate Carrée** (palavra francesa para "placa quadrada") é um tipo específico de **projeção cilíndrica equidistante**. Ela mapeia a latitude e a longitude diretamente em um sistema de eixos retangulares Cartesianos $X$ e $Y$.

```
Latitude   (+90°)  ┌──────────────────────────────┐
                   │                              │
           (0°)    │          A L T O             │  <- Cada pixel representa um
                   │         O C E A N O          │     mesmo tamanho em graus
                   │                              │     (ex: 0.0416° x 0.0416°)
           (-90°)  └──────────────────────────────┘
                 (-180°)        (0°)          (+180°)
                               Longitude

```

### Características Técnicas:

* **Eixo X:** Longitude (varia de $-180^\circ$ a $+180^\circ$).
* **Eixo Y:** Latitude (varia de $-90^\circ$ a $+90^\circ$).
* **Grade Regular em Graus (Equal Angle Grid):** O tamanho de cada pixel no arquivo é fixo em termos de **graus geográficos** (ex: $0,04166^\circ \times 0,04166^\circ$).
* **Referência de Datum:** Utiliza o modelo geométrico terrestre **WGS84** (EPSG:4326).

### Vantagens e Limitações

| Vantagens | Limitações |
| --- | --- |
| **Simplicidade de Indexação:** A relação entre o índice da linha/coluna na matriz de dados e as coordenadas de Latitude/Longitude é puramente linear. | **Distorção de Área nas Altas Latitudes:** Como os meridianos não convergem no mapa, as regiões próximas aos polos aparecem muito esticadas horizontalmente. |
| **Compatibilidade Universal:** É o padrão global mais aceito para integração em Python, sistemas GIS e plataformas de computação em nuvem. | **Variação da Resolução em Metros:** A resolução de `3000` (ou ~4,6 km) vale exatamente na linha do Equador; conforme você se aproxima dos polos, a largura do pixel em quilômetros diminui. |


### 2.2.2 -  Nomenclatura dos arquivos netcdf

exemplo
```
GC1SG1_20180101A08D_D0000_3MSG_CHLAM_3000.nc
```

| Bloco | Exemplo | Significado |
| :--- | :--- | :--- |
| **Plataforma e Sensor** | `GC1SG1` | **GC1**: Satélite GCOM-C1 (SHIKISAI).<br>**SG1**: Sensor SGLI (*Second Generation Global Imager*). |
| **Data e Período** | `20180101A08D` | **20180101**: Data inicial (AAAAMMDD).<br>**A** / **D**: Órbita **A**scendente (Diurna) ou **D**escendente (Noturna).<br>**08D**: Média composta de 08 dias. |
| **Versão** | `D0000` | Código da versão do algoritmo e calibração da JAXA. |
| **Nível do Produto** | `3MSG` | **3**: Nível 3 (L3).<br>**M**: Média estatística.<br>**SG**: Projeção geográfica global (*Plate Carrée*). |
| **Variável** | `CHLAM` | **CHLA**: Clorofila-a.<br>**M**: Média (*Mean*) dos valores do período. |
| **Resolução Espacial** | `3000` | Resolução do pixel (~3000m / 4,6 km no equador). |
| **Extensão** | `.nc` | Formato do arquivo NetCDF-4. |

In [ ]:
import ftplib
import os
from tqdm import tqdm

#region VARIÁVEIS_CONFIGURAÇÃO


FTP_HOST = "apollo.eorc.jaxa.jp"

FTP_USER = "anonymous"

FTP_PASS = "miguel.penteado@ibge.gov.br"

REMOTE_BASE_DIR = "/pub/SGLI_STD/Global_05km/8-day/CHLA"

LOCAL_BASE_DIR = "./downloads_jaxa"

EXTENSAO_DOWNLOAD = ".nc"

 #endregion

#region ESTABELECER_CONEXÃO_FTP

def conectar_ftp():
    """
    Conecta ao servidor FTP e realiza o login.

    Retorna uma instância de ftplib.FTP.
    """

    print(f"Conectando ao servidor FTP " f"{FTP_HOST}...")

    ftp = ftplib.FTP(FTP_HOST)

    ftp.login(user=FTP_USER, passwd=FTP_PASS)

    print("Conexão estabelecida com sucesso!")

    return ftp

#endregion

#region TAMANHO_DO_ARQUIVO

def obter_tamanho_arquivo(ftp, remote_filepath):
    """
    Obtém o tamanho de um arquivo remoto.

    Alguns servidores FTP podem não suportar
    o comando SIZE. Nesse caso, retorna None.
    """

    try:

        return ftp.size(remote_filepath)

    except Exception:

        return None

#endregion

#region DOWNLOAD_UM_ARQUIVO

def download_file(ftp, remote_filepath, local_filepath):
    """
    Baixa um único arquivo via FTP exibindo
    uma barra de progresso.
    """

    # --------------------------------------------------------
    # Cria diretório local
    # --------------------------------------------------------

    diretorio_local = os.path.dirname(local_filepath)

    os.makedirs(diretorio_local, exist_ok=True)

    # --------------------------------------------------------
    # Remove arquivo incompleto anterior
    # --------------------------------------------------------

    if os.path.exists(local_filepath):

        os.remove(local_filepath)

    # --------------------------------------------------------
    # Tamanho remoto
    # --------------------------------------------------------

    total_size = obter_tamanho_arquivo(ftp, remote_filepath)

    filename = os.path.basename(remote_filepath)

    # --------------------------------------------------------
    # Barra de progresso
    # --------------------------------------------------------

    progress_bar = tqdm(
        total=total_size,
        unit="B",
        unit_scale=True,
        unit_divisor=1024,
        desc=filename,
        leave=False,
    )

    # --------------------------------------------------------
    # Callback
    # --------------------------------------------------------

    with open(local_filepath, "wb") as arquivo_local:

        def callback(data):
            """
            Callback chamado pelo retrbinary
            a cada bloco recebido.
            """

            arquivo_local.write(data)

            progress_bar.update(len(data))

        try:

            ftp.retrbinary(f"RETR {remote_filepath}", callback)

        finally:

            progress_bar.close()

#endregion

#region LISTAGEM_DO_DIRETÓRIO

def listar_itens_remotos(ftp, remote_dir):
    """
    Lista os arquivos e diretórios existentes
    em um diretório remoto.

    Retorna uma lista vazia em caso de erro
    de permissão ou diretório vazio.
    """

    try:

        return ftp.nlst(remote_dir)

    except ftplib.error_perm:

        return []

#endregion

#region TESTE_DE_DIRETÓRIO

def eh_diretorio(ftp, remote_path):
    """
    Verifica se um caminho remoto representa
    um diretório.

    Retorna True ou False.
    """

    diretorio_atual = ftp.pwd()

    try:

        ftp.cwd(remote_path)

        return True

    except ftplib.error_perm:

        return False

    finally:

        try:

            ftp.cwd(diretorio_atual)

        except ftplib.error_perm:

            pass

#endregion

#region DOWNLOAD_RECURSIVO

def download_recursive(ftp, remote_dir, local_dir):
    """
    Percorre recursivamente os diretórios do FTP
    e baixa arquivos NetCDF.
    """

    items = listar_itens_remotos(ftp, remote_dir)

    if not items:

        return

    for item in items:

        # ----------------------------------------------------
        # Evita referências especiais
        # ----------------------------------------------------

        if item.endswith("/.") or item.endswith("/.."):

            continue

        item_name = os.path.basename(item.rstrip("/"))

        local_path = os.path.join(local_dir, item_name)

        # ----------------------------------------------------
        # Diretório
        # ----------------------------------------------------

        if eh_diretorio(ftp, item):

            os.makedirs(local_path, exist_ok=True)

            download_recursive(ftp, item, local_path)

            continue

        # ----------------------------------------------------
        # Arquivo
        # ----------------------------------------------------

        if item.lower().endswith(EXTENSAO_DOWNLOAD):

            print(f"Baixando: {item}")

            download_file(ftp, item, local_path)

#endregion

#region ENCERRAMENTO

def encerrar_conexao(ftp):
    """
    Encerra a conexão FTP.
    """

    try:

        ftp.quit()

    except Exception:

        try:
            ftp.close()

        except Exception:
            pass

#endregion

#region FUNÇÃO_PRINCIPAL

def main():
    """
    Função principal do programa.
    """

    ftp = None

    try:

        # ----------------------------------------------------
        # 1. Conecta
        # ----------------------------------------------------

        ftp = conectar_ftp()

        # ----------------------------------------------------
        # 2. Cria diretório local
        # ----------------------------------------------------

        os.makedirs(LOCAL_BASE_DIR, exist_ok=True)

        # ----------------------------------------------------
        # 3. Download recursivo
        # ----------------------------------------------------

        download_recursive(ftp, REMOTE_BASE_DIR, LOCAL_BASE_DIR)

        # ----------------------------------------------------
        # 4. Resultado
        # ----------------------------------------------------

        print()

        print("Download concluído com sucesso!")

    except Exception as erro:

        print()

        print("Erro durante o processamento:")

        print(f"  {erro}")

    finally:

        if ftp is not None:

            encerrar_conexao(ftp)

#endregion

# ============================================================
# PONTO DE ENTRADA
# ============================================================

if __name__ == "__main__":
    main()

'''
main()
│
├── conectar_ftp()
├── download_recursive()
│   ├── listar_itens_remotos()
│   ├── eh_diretorio()
│   └── download_file()
└── encerrar_conexao()
'''

## 2.3 Converter as imagens NetCDF para GeoTiff e projeção latitude de 0 - 360 para -180 a +180

In [ ]:
import glob
import os

from osgeo import gdal

gdal.UseExceptions()

#region VARIÁVEIS_GLOBAIS_CONFIGURAÇÃO

DIRETORIO = "."

EXTENSAO_ENTRADA = "*.nc"

LARGURA = 7200
ALTURA = 3601

SRC_SRS = "+proj=longlat +ellps=WGS84 +lon_wrap=180"
DST_SRS = "EPSG:4326"

OUTPUT_BOUNDS = [ -180, -90, 180, 90, ]

#endregion

#region BUSCA_ARQUIVOS

def localizar_netcdfs(diretorio):
    """
    Localiza recursivamente todos os arquivos NetCDF
    existentes no diretório informado.
    """

    arquivos = glob.glob( os.path.join(diretorio, "**", EXTENSAO_ENTRADA), recursive=True )

    return sorted(set(arquivos))

#endregion

#region SUBDATASET

def obter_primeira_subdataset(arquivo):
    """
    Abre um NetCDF e retorna o nome da primeira subdataset.

    Exemplo de retorno:

    NETCDF:"arquivo.nc":CHLA_AVE
    """

    dataset = gdal.Open(arquivo)

    if dataset is None:
        raise RuntimeError(f"Não foi possível abrir o NetCDF: {arquivo}")

    subdatasets = dataset.GetSubDatasets()

    dataset = None

    if not subdatasets:
        raise RuntimeError("Nenhuma subdataset encontrada.")

    return subdatasets[0][0]

#endregion

#region METADADOS_RADIOMÉTRICOS

def obter_metadados_radiometricos(dataset):
    """
    Obtém os principais metadados radiométricos
    da primeira banda do dataset.
    """

    banda = dataset.GetRasterBand(1)

    return {
        "scale": banda.GetScale(),
        "offset": banda.GetOffset(),
        "nodata": banda.GetNoDataValue(),
        "datatype": gdal.GetDataTypeName(banda.DataType),
    }

#endregion

#region NOME_DO_ARQUIVO_DE_SAÍDA

def obter_arquivo_saida(arquivo_entrada):
    """
    Substitui a extensão .nc por .tif.
    """

    return os.path.splitext(arquivo_entrada)[0] + ".tif"

#endregion

#region CONVERSÃO_PARA_GEOTIFF

def converter_para_geotiff(src, arquivo_saida):
    """
    Converte o dataset para GeoTIFF usando gdal.Warp.

    A longitude é reorganizada de 0-360 para -180/+180.
    """

    warp_options = gdal.WarpOptions(
        format="GTiff",
        srcSRS=SRC_SRS,
        dstSRS=DST_SRS,
        outputBounds=OUTPUT_BOUNDS,
        width=LARGURA,
        height=ALTURA,
        creationOptions=[
            "COMPRESS=DEFLATE",
            "TILED=YES",
        ],
    )

    destino = gdal.Warp(arquivo_saida, src, options=warp_options)

    if destino is None:
        raise RuntimeError("Erro durante o gdal.Warp.")

    destino = None

#endregion

#region METADADOS_DO_GEOTIFF

def gravar_metadados_geotiff(arquivo_saida, arquivo_origem, subdataset_name, metadados):
    """
    Reabre o GeoTIFF e grava scale, offset
    e informações de proveniência.
    """

    tif = gdal.Open(arquivo_saida, gdal.GA_Update)

    if tif is None:
        raise RuntimeError("Erro ao reabrir o GeoTIFF.")

    banda = tif.GetRasterBand(1)

    scale = metadados["scale"]

    # --------------------------------------------------------
    # Scale
    # --------------------------------------------------------

    if scale is not None:

        banda.SetScale(scale)

        banda.SetMetadataItem("SCALE_FACTOR", str(scale))

    # --------------------------------------------------------
    # Offset
    # --------------------------------------------------------

    if (offset := metadados["offset"]) is not None:

        banda.SetOffset(offset)

        banda.SetMetadataItem("ADD_OFFSET", str(offset))

    # --------------------------------------------------------
    # Proveniência
    # --------------------------------------------------------

    banda.SetMetadataItem("SOURCE_FILE", os.path.basename(arquivo_origem))

    banda.SetMetadataItem("SOURCE_SUBDATASET", subdataset_name)

    banda.SetMetadataItem("PROCESSING", "Longitude converted from 0-360 to -180/+180")

    # --------------------------------------------------------
    # Gravação
    # --------------------------------------------------------

    banda.FlushCache()

    tif.FlushCache()

    tif = None

#endregion

#region PROCESSAMENTO_DE_UM_ARQUIVO

def processar_arquivo(arquivo):
    """
    Processa um único arquivo NetCDF.

    Retorna True em caso de sucesso e False em caso de erro.
    """

    print("=" * 80)
    print(f"Processando: {arquivo}")

    try:

        # ----------------------------------------------------
        # 1. Obtém subdataset
        # ----------------------------------------------------

        subdataset_name = obter_primeira_subdataset(arquivo)

        print("Subdataset selecionada:")

        print(f"  {subdataset_name}")

        # ----------------------------------------------------
        # 2. Abre subdataset
        # ----------------------------------------------------

        src = gdal.Open(subdataset_name)

        if src is None:
            raise RuntimeError("Erro ao abrir a subdataset.")

        # ----------------------------------------------------
        # 3. Metadados
        # ----------------------------------------------------

        metadados = obter_metadados_radiometricos(src)

        print("\nMetadados radiométricos:")

        print(f"  Scale  = {metadados['scale']}")

        print(f"  Offset = {metadados['offset']}")

        print(f"  NoData = {metadados['nodata']}")

        print(f"  Tipo   = {metadados['datatype']}")

        # ----------------------------------------------------
        # 4. Arquivo de saída
        # ----------------------------------------------------

        arquivo_saida = obter_arquivo_saida(arquivo)

        print("\nConversão:")

        print(f"  Entrada : {arquivo}")

        print(f"  Saída   : {arquivo_saida}")

        # ----------------------------------------------------
        # 5. Conversão
        # ----------------------------------------------------

        converter_para_geotiff(src, arquivo_saida)

        src = None

        # ----------------------------------------------------
        # 6. Metadados no GeoTIFF
        # ----------------------------------------------------

        gravar_metadados_geotiff(arquivo_saida, arquivo, subdataset_name, metadados)

        # ----------------------------------------------------
        # Resultado
        # ----------------------------------------------------

        print("\nMetadados gravados no GeoTIFF:")

        print(f"  Scale  = {metadados['scale']}")

        print(f"  Offset = {metadados['offset']}")

        print(f"\nSucesso: {arquivo_saida}\n")

        return True

    except Exception as erro:

        print("\nErro ao processar:")

        print(f"  {arquivo}")

        print(f"  {erro}\n")

        return False

#endregion

#region RESUMO

def imprimir_resumo_final(total_arquivos, total_sucessos, total_erros):
    """
    Imprime o resumo final do processamento.
    """

    print("=" * 80)
    print("PROCESSO CONCLUÍDO")
    print("=" * 80)

    print(f"Arquivos encontrados : {total_arquivos}")

    print(f"Convertidos          : {total_sucessos}")

    print(f"Erros                : {total_erros}")

    print("=" * 80)

#endregion

#region FUNÇÃO_PRINCIPAL

def main():
    """
    Função principal do programa.
    """

    # --------------------------------------------------------
    # 1. Localiza arquivos
    # --------------------------------------------------------

    arquivos_nc = localizar_netcdfs(DIRETORIO)

    if not arquivos_nc:

        print("Nenhum arquivo .nc encontrado " "no diretório ou subdiretórios.")

        return

    print(f"Encontrados " f"{len(arquivos_nc)} " f"arquivos para conversão.\n")

    # --------------------------------------------------------
    # 2. Contadores
    # --------------------------------------------------------

    total_sucessos = 0
    total_erros = 0

    # --------------------------------------------------------
    # 3. Processamento
    # --------------------------------------------------------

    for arquivo in arquivos_nc:

        sucesso = processar_arquivo(arquivo)

        if sucesso:
            total_sucessos += 1

        else:
            total_erros += 1

    # --------------------------------------------------------
    # 4. Resumo
    # --------------------------------------------------------

    imprimir_resumo_final(len(arquivos_nc), total_sucessos, total_erros)

#endregion

# ============================================================
# PONTO DE ENTRADA
# ============================================================

if __name__ == "__main__":
    main()

'''
main()
│
├── localizar_netcdfs()
│
├── processar_arquivo()
│   ├── obter_primeira_subdataset()
│   ├── obter_metadados_radiometricos()
│   ├── converter_para_geotiff()
│   └── gravar_metadados_geotiff()
│
└── imprimir_resumo_final()
'''


## 2.4 Obter os polígonos do CNUC (Cadastro Nacional de Unidades de Conservação) do Ministério do Meio Ambiente

Você deve fazer download do arquivo (arquivo compactado) **ucs.zip** no site https://cnuc.mma.gov.br/map

In [ ]:
import requests

url = "https://cnuc-backend.mma.gov.br/api/v1/downloadGeo"

headers = {
    "Accept": "application/json, text/plain, */*",
    "Accept-Language": "pt-BR,pt;q=0.7",
    "Connection": "keep-alive",
    "Content-Type": "application/json;charset=UTF-8",
    "Email": "null",
    "IdUser": "null",
    "Origin": "https://cnuc.mma.gov.br",
    "Referer": "https://cnuc.mma.gov.br/",
    "Sec-Fetch-Dest": "empty",
    "Sec-Fetch-Mode": "cors",
    "Sec-Fetch-Site": "same-site",
    "Sec-GPC": "1",
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/154.0.0.0 Safari/537.36",
    "panel": "null",
    "sec-ch-ua": '"Chromium";v="154", "Brave";v="154", "Not A(Brand";v="99"',
    "sec-ch-ua-mobile": "?0",
    "sec-ch-ua-platform": '"Windows"',
    "sphere": "null",
    "uf": "null",
}

payload = {
    "format": "shp",
    "map": "/var/www/storage/app/mapfiles/ucs.map",
    "name": "ucs",
    "typename": "ucs_selected",
    "ucIds": "null"
}

# Realiza a requisição POST enviando o JSON
response = requests.post(url, headers=headers, json=payload)

# Salva o resultado no arquivo arquivo teste.zip
if response.status_code == 200:
    with open("cnuc_unidades_de_conservacao.zip", "wb") as f:
        f.write(response.content)
    print("Download concluído com sucesso!")
else:
    print(f"Erro na requisição: {response.status_code}")
    print(response.text)

### 2.4.1 Descompacte o arquivo zipado
Extraia o conteúdo do arquivo zipado **cnuc_unidades_de_conservacao.zip** para a pasta  ""camada_vetorial_cnuc""

In [ ]:
import os
import zipfile

# Nome do arquivo ZIP e da pasta de destino
zip_filename = "cnuc_unidades_de_conservacao.zip"
output_folder = "camada_vetorial_cnuc"

# Cria a pasta caso ela não exista
os.makedirs(output_folder, exist_ok=True)

# Extrai o conteúdo do arquivo ZIP
with zipfile.ZipFile(zip_filename, 'r') as zip_ref:
    zip_ref.extractall(output_folder)

print(f"Conteúdo extraído com sucesso para a pasta '{output_folder}'.")

## 2.5 Reprojetar o dataset vetorial de entrada de Unidades de Conservação do CNUC Sirgas 2000 para WGS 84

In [ ]:
!ogr2ogr   -f "ESRI Shapefile"   -s_srs EPSG:4674   -t_srs EPSG:4326  -lco ENCODING=UTF-8 -nln ucs   camada_vetorial_cnuc\ucs4326.shp   camada_vetorial_cnuc\ucs.shp

## 2.6 Fazer o recorte das UCs Marinhas

In [ ]:
import glob
import os
import re

from osgeo import gdal, ogr, osr

gdal.UseExceptions()


# region VARIAVEIS_GLOBAIS_CONFIGURAÇÃO


DIR_RASTERS = r".\downloads_jaxa"
ARQUIVO_UCS = r".\camada_vetorial_cnuc\ucs.shp"
DIR_SAIDA = r".\raster_ucs"

ARQUIVO_UCS_MARINHAS = os.path.join(DIR_SAIDA, "ucs_marinhas_epsg4326_temporario.gpkg")

EXTENSOES = [ "*.tif", "*.tiff", ]

CAMPOS_COPIAR = [
    "gml_id",
    "docleg_id",
    "uc_id",
    "cd_cnuc",
    "wdpa_pid",
    "nome_uc",
    "cria_ano",
    "cria_ato",
    "outro_ato",
    "pl_manejo",
    "co_gestor",
    "quali_pol",
    "ppgr",
    "ha_total",
    "ha_ato",
    "esfera",
    "uf",
    "municipio",
    "org_gestor",
    "grupo",
    "categoria",
    "cat_iucn",
    "amazonia",
    "caatinga",
    "cerrado",
    "matlantica",
    "pampa",
    "pantanal",
    "marinho",
    "situacao",
    "dt_cert_og",
    "limite",
]

# endregion


# region FUNÇÕES_AUXILIARES

def limpar_nome(texto):
    """
    Remove caracteres inadequados para nomes de arquivo.
    """

    if texto is None:
        return "SEM_VALOR"

    texto = str(texto).strip()

    substituicoes = {
        "á": "a",
        "à": "a",
        "ã": "a",
        "â": "a",
        "é": "e",
        "ê": "e",
        "í": "i",
        "ó": "o",
        "ô": "o",
        "õ": "o",
        "ú": "u",
        "ç": "c",
        "Á": "A",
        "À": "A",
        "Ã": "A",
        "Â": "A",
        "É": "E",
        "Ê": "E",
        "Í": "I",
        "Ó": "O",
        "Ô": "O",
        "Õ": "O",
        "Ú": "U",
        "Ç": "C",
    }

    for origem, destino in substituicoes.items():
        texto = texto.replace(origem, destino)

    texto = re.sub(r"[^A-Za-z0-9_-]+", "_", texto)

    texto = re.sub(r"_+", "_", texto)

    return texto.strip("_")


def bbox_intersecta(a, b):
    """
    Testa se dois bounding boxes se interceptam.

    Formato:
    (xmin, ymin, xmax, ymax)
    """

    return not (a[2] < b[0] or a[0] > b[2] or a[3] < b[1] or a[1] > b[3])


def obter_extent_raster(dataset):
    """
    Obtém o bounding box de um raster.

    Retorno:
    (xmin, ymin, xmax, ymax)
    """

    gt = dataset.GetGeoTransform()

    largura = dataset.RasterXSize
    altura = dataset.RasterYSize

    cantos_pixel = [ (0, 0), (largura, 0), (0, altura), (largura, altura), ]

    pontos = [gdal.ApplyGeoTransform(gt, x, y) for x, y in cantos_pixel]

    xs = [p[0] for p in pontos]
    ys = [p[1] for p in pontos]

    return ( min(xs), min(ys), max(xs), max(ys), )

# endregion


# region FUNÇÕES_SISTEMAS_DE_REFERENCIA

def criar_srs(epsg):
    """
    Cria um SpatialReference usando a ordem GIS tradicional:

    X = longitude
    Y = latitude
    """

    srs = osr.SpatialReference()

    srs.ImportFromEPSG(epsg)

    srs.SetAxisMappingStrategy(osr.OAMS_TRADITIONAL_GIS_ORDER)

    return srs

# endregion


#region PREPARAR_UCS
'''============================================================'''


def preparar_ucs_marinhas():
    """
    Lê o CNUC, seleciona as UCs marinhas, reprojeta para
    EPSG:4326 e cria um GeoPackage temporário.

    Retorna uma lista contendo informações das UCs.
    """

    print("=" * 80)
    print("Preparando camada de UCs marinhas...")
    print("=" * 80)

    remover_gpkg_temporario()

    src_ds = ogr.Open(ARQUIVO_UCS)

    if src_ds is None:
        raise RuntimeError(f"Não foi possível abrir: {ARQUIVO_UCS}")

    src_layer = src_ds.GetLayer()

    # --------------------------------------------------------
    # CRS
    # --------------------------------------------------------

    src_srs = criar_srs(4674)

    dst_srs = criar_srs(4326)

    print()
    print("SRC axis mapping:", src_srs.GetDataAxisToSRSAxisMapping())

    print("DST axis mapping:", dst_srs.GetDataAxisToSRSAxisMapping())
    print()

    transformacao = osr.CoordinateTransformation(src_srs, dst_srs)

    # --------------------------------------------------------
    # GeoPackage
    # --------------------------------------------------------

    driver_gpkg = ogr.GetDriverByName("GPKG")

    dst_ds = driver_gpkg.CreateDataSource(ARQUIVO_UCS_MARINHAS)

    if dst_ds is None:
        raise RuntimeError("Não foi possível criar o GeoPackage temporário.")

    dst_layer = dst_ds.CreateLayer(
        "ucs_marinhas", srs=dst_srs, geom_type=ogr.wkbMultiPolygon
    )

    criar_campos_destino(src_layer, dst_layer)

    ucs = extrair_ucs_marinhas(src_layer, dst_layer, transformacao)

    dst_layer.SyncToDisk()

    dst_ds = None
    src_ds = None

    print()
    print(f"UCs marinhas encontradas: {len(ucs)}")
    print()

    return ucs


def remover_gpkg_temporario():
    """
    Remove o GeoPackage temporário caso exista.
    """

    if not os.path.exists(ARQUIVO_UCS_MARINHAS):
        return

    driver = ogr.GetDriverByName("GPKG")

    driver.DeleteDataSource(ARQUIVO_UCS_MARINHAS)


def criar_campos_destino(src_layer, dst_layer):
    """
    Cria no GeoPackage temporário os campos necessários
    para o recorte.
    """

    campo_clip = ogr.FieldDefn("clip_id", ogr.OFTInteger)

    dst_layer.CreateField(campo_clip)

    src_defn = src_layer.GetLayerDefn()

    for nome_campo in CAMPOS_COPIAR:

        idx = src_defn.GetFieldIndex(nome_campo)

        if idx < 0:
            continue

        campo_origem = src_defn.GetFieldDefn(idx)

        novo_campo = ogr.FieldDefn(nome_campo, campo_origem.GetType())

        novo_campo.SetWidth(campo_origem.GetWidth())

        dst_layer.CreateField(novo_campo)


def extrair_ucs_marinhas(src_layer, dst_layer, transformacao):
    """
    Seleciona as UCs com atributo 'marinho' preenchido,
    reprojeta suas geometrias e grava no GeoPackage.
    """

    ucs = []

    contador = 0

    src_layer.ResetReading()

    for feature in src_layer:

        marinho = feature.GetField("marinho")

        if marinho is None:
            continue

        if str(marinho).strip() == "":
            continue

        geom = feature.GetGeometryRef()

        if geom is None:
            continue

        geom = geom.Clone()

        geom.Transform(transformacao)

        geom = ogr.ForceToMultiPolygon(geom)

        contador += 1

        nova_feature = ogr.Feature(dst_layer.GetLayerDefn())

        nova_feature.SetField("clip_id", contador)

        copiar_atributos(feature, nova_feature)

        nova_feature.SetGeometry(geom)

        dst_layer.CreateFeature(nova_feature)

        env = geom.GetEnvelope()

        bbox = ( env[0], env[2], env[1], env[3], )

        ucs.append(
            {
                "clip_id": contador,
                "uc_id": feature.GetField("uc_id"),
                "cd_cnuc": feature.GetField("cd_cnuc"),
                "nome_uc": feature.GetField("nome_uc"),
                "bbox": bbox,
            }
        )

        nova_feature = None

    return ucs


def copiar_atributos(feature_origem, feature_destino):
    """
    Copia os atributos definidos em CAMPOS_COPIAR.
    """

    for campo in CAMPOS_COPIAR:

        idx = feature_origem.GetFieldIndex(campo)

        if idx < 0:
            continue

        valor = feature_origem.GetField(campo)

        if valor is not None:

            feature_destino.SetField(campo, valor)

# endregion


#region LOCALIZAÇÃO_DOS_RASTERS


def localizar_rasters():
    """
    Localiza recursivamente todos os GeoTIFFs da pasta
    DIR_RASTERS.
    """

    arquivos = []

    for extensao in EXTENSOES:

        arquivos.extend( glob.glob(os.path.join(DIR_RASTERS, "**", extensao), recursive=True))

    arquivos = sorted(set(arquivos))

    print("=" * 80)
    print(f"Rasters encontrados: {len(arquivos)}")
    print("=" * 80)
    print()

    return arquivos

#endregion


#region ORGANIZAÇÃO_DOS_RASTERS



def obter_ano_raster(arquivo_raster):
    """
    Identifica o ano com base no primeiro diretório relativo
    a DIR_RASTERS.
    """

    relativo = os.path.relpath(arquivo_raster, DIR_RASTERS)

    partes = relativo.split(os.sep)

    if len(partes) > 1:
        return partes[0]

    return "sem_ano"


def obter_diretorio_saida_ano(arquivo_raster):
    """
    Cria e retorna a pasta correspondente ao ano do raster.
    """

    ano = obter_ano_raster(arquivo_raster)

    dir_saida_ano = os.path.join(DIR_SAIDA, ano)

    os.makedirs(dir_saida_ano, exist_ok=True)

    return dir_saida_ano

#endregion


#region METADADOS

def obter_metadados_raster(raster):
    """
    Obtém os principais metadados científicos da banda 1.
    """

    banda = raster.GetRasterBand(1)

    return {
        "scale": banda.GetScale(),
        "offset": banda.GetOffset(),
        "nodata": banda.GetNoDataValue(),
        "metadata_banda": banda.GetMetadata(),
    }

def gravar_metadados_saida(arquivo_saida, arquivo_raster, uc, metadados):
    """
    Reabre o GeoTIFF gerado e grava os metadados
    científicos e de proveniência.
    """

    tif = gdal.Open(arquivo_saida, gdal.GA_Update)

    if tif is None:
        return

    banda = tif.GetRasterBand(1)

    scale = metadados["scale"]
    offset = metadados["offset"]
    nodata = metadados["nodata"]

    # --------------------------------------------------------
    # Scale
    # --------------------------------------------------------

    if scale is not None:

        banda.SetScale(scale)

        banda.SetMetadataItem("SCALE_FACTOR", str(scale))

    # --------------------------------------------------------
    # Offset
    # --------------------------------------------------------

    if offset is not None:

        banda.SetOffset(offset)

        banda.SetMetadataItem("ADD_OFFSET", str(offset))

    # --------------------------------------------------------
    # NoData
    # --------------------------------------------------------

    if nodata is not None:

        banda.SetNoDataValue(nodata)

    # --------------------------------------------------------
    # Metadados originais
    # --------------------------------------------------------

    if metadata_banda := metadados["metadata_banda"]:

        for chave, valor in metadata_banda.items():

            banda.SetMetadataItem(chave, str(valor))

    # --------------------------------------------------------
    # Proveniência
    # --------------------------------------------------------

    tif.SetMetadataItem("SOURCE_RASTER", os.path.basename(arquivo_raster))

    tif.SetMetadataItem("SOURCE_VECTOR", "CNUC")

    tif.SetMetadataItem("CNUC_CODE", str(uc["cd_cnuc"]))

    tif.SetMetadataItem("UC_NAME", str(uc["nome_uc"]))

    tif.SetMetadataItem("PROCESSING", "Raster clipped by CNUC marine conservation unit")

    tif.SetMetadataItem("OUTPUT_FILENAME_PATTERN", "original-raster-name-cd_cnuc.tif")

    banda.FlushCache()
    tif.FlushCache()

    tif = None

#endregion


#region RECORTE

def recortar_uc(raster, arquivo_raster, arquivo_saida, uc, metadados):
    """
    Executa o recorte de um raster usando uma UC como cutline.
    """

    where = f"clip_id = {uc['clip_id']}"

    warp_options = gdal.WarpOptions(
        format="GTiff",
        cutlineDSName=ARQUIVO_UCS_MARINHAS,
        cutlineLayer="ucs_marinhas",
        cutlineWhere=where,
        cropToCutline=True,
        dstSRS="EPSG:4326",
        xRes=0.05,
        yRes=0.05,
        targetAlignedPixels=True,
        dstNodata=metadados["nodata"],
        resampleAlg="near",
        multithread=True,
        creationOptions=[
            "COMPRESS=DEFLATE",
            "TILED=YES",
            "BIGTIFF=IF_SAFER",
        ],
    )

    recorte = gdal.Warp(arquivo_saida, raster, options=warp_options)

    if recorte is None:
        raise RuntimeError("gdal.Warp retornou None.")

    recorte = None

    gravar_metadados_saida(arquivo_saida, arquivo_raster, uc, metadados)

#endregion


#region PROCESSAMENTO_DE_UM_RASTER

def processar_raster(arquivo_raster, ucs, indice, total_rasters):
    """
    Processa um único raster contra todas as UCs marinhas.

    Retorna contadores locais.
    """

    resultado = {
        "recortes": 0,
        "existentes": 0,
        "erros": 0,
        "sem_intersecao": 0,
    }

    print()
    print("=" * 80)

    print(f"Raster {indice}/{total_rasters}")

    print(f"Arquivo: {arquivo_raster}")

    print("=" * 80)

    try:

        raster = gdal.Open(arquivo_raster, gdal.GA_ReadOnly)

        if raster is None:

            print("Não foi possível abrir o raster.")

            resultado["erros"] += 1

            return resultado

        # ----------------------------------------------------
        # Metadados
        # ----------------------------------------------------

        metadados = obter_metadados_raster(raster)

        print(f"Scale : {metadados['scale']}")

        print(f"Offset: {metadados['offset']}")

        print(f"NoData: {metadados['nodata']}")

        # ----------------------------------------------------
        # Extent
        # ----------------------------------------------------

        bbox_raster = obter_extent_raster(raster)

        print(f"Extent raster: {bbox_raster}")

        # ----------------------------------------------------
        # Pasta e nome
        # ----------------------------------------------------

        dir_saida_ano = obter_diretorio_saida_ano(arquivo_raster)

        nome_raster = os.path.splitext(os.path.basename(arquivo_raster))[0]

        # ----------------------------------------------------
        # UCs
        # ----------------------------------------------------

        for uc in ucs:

            if not bbox_intersecta(bbox_raster, uc["bbox"]):

                resultado["sem_intersecao"] += 1
                continue

            codigo = uc["cd_cnuc"]

            if codigo is None or str(codigo).strip() == "":

                codigo = f"SEM_CD_CNUC_{uc['clip_id']}"

            codigo = limpar_nome(codigo)

            nome_saida = f"{nome_raster}" f"-{codigo}.tif"

            arquivo_saida = os.path.join(dir_saida_ano, nome_saida)

            # ------------------------------------------------
            # Arquivo já existente
            # ------------------------------------------------

            if os.path.exists(arquivo_saida):

                print(f"Já existe: {nome_saida}")

                resultado["existentes"] += 1

                continue

            # ------------------------------------------------
            # Recorte
            # ------------------------------------------------

            try:

                recortar_uc(raster, arquivo_raster, arquivo_saida, uc, metadados)

                resultado["recortes"] += 1

                print(f"OK: {nome_saida}")

            except Exception as erro:

                resultado["erros"] += 1

                print(f"ERRO UC " f"{uc['nome_uc']} " f"({codigo}): " f"{erro}")

        raster = None

    except Exception as erro:

        resultado["erros"] += 1

        print(f"Erro no raster " f"{arquivo_raster}: " f"{erro}")

    return resultado

#endregion


#region PROCESSAMENTO_DE_TODOS_OS_RASTERS

def processar_rasters(arquivos_raster, ucs):
    """
    Processa todos os rasters encontrados.
    """

    totais = {
        "recortes": 0,
        "existentes": 0,
        "erros": 0,
        "sem_intersecao": 0,
    }

    total_rasters = len(arquivos_raster)

    for indice, arquivo_raster in enumerate(arquivos_raster, start=1):

        resultado = processar_raster(arquivo_raster, ucs, indice, total_rasters)

        for chave in totais:
            totais[chave] += resultado[chave]

    return totais

#endregion


#region RESULTADO_FINAL

def imprimir_resumo(arquivos_raster, ucs, totais):
    """
    Exibe o resumo final do processamento.
    """

    print()
    print("=" * 80)
    print("PROCESSAMENTO CONCLUÍDO")
    print("=" * 80)

    print(f"Rasters de entrada       : " f"{len(arquivos_raster)}")

    print(f"UCs marinhas             : " f"{len(ucs)}")

    print(f"Recortes gerados         : " f"{totais['recortes']}")

    print(f"Já existentes            : " f"{totais['existentes']}")

    print(f"Erros                    : " f"{totais['erros']}")

    print(f"Testes sem intersecção   : " f"{totais['sem_intersecao']}")

    print()

    print(f"Resultados em: " f"{DIR_SAIDA}")

    print("=" * 80)

#endregion


#region FUNÇÃO_PRINCIPAL

def main():
    """
    Função principal do programa.
    """

    # --------------------------------------------------------
    # 1. Prepara ambiente
    # --------------------------------------------------------

    os.makedirs(DIR_SAIDA, exist_ok=True)

    # --------------------------------------------------------
    # 2. Prepara camada vetorial
    # --------------------------------------------------------

    ucs = preparar_ucs_marinhas()

    # --------------------------------------------------------
    # 3. Localiza os rasters
    # --------------------------------------------------------

    arquivos_raster = localizar_rasters()

    # --------------------------------------------------------
    # 4. Processa os rasters
    # --------------------------------------------------------

    totais = processar_rasters(arquivos_raster, ucs)

    # --------------------------------------------------------
    # 5. Relatório
    # --------------------------------------------------------

    imprimir_resumo(arquivos_raster, ucs, totais)

#endregion


#region PONTO_DE_ENTRADA

if __name__ == "__main__":
    main()

#endregion


"""
main()
│
├── preparar_ambiente()
│
├── preparar_ucs_marinhas()
│   ├── criar_srs()
│   └── copiar_campos()
│
├── localizar_rasters()
│
├── processar_rasters()
│   └── processar_raster()
│       ├── obter_extent_raster()
│       ├── bbox_intersecta()
│       └── recortar_uc()
│           └── gravar_metadados()
│
└── imprimir_resumo()
"""

## 2.7 -  Criação dos Mosaicos

In [ ]:
import os
import glob

from collections import defaultdict
from osgeo import gdal


gdal.UseExceptions()


# ============================================================
# CONFIGURAÇÃO
# ============================================================

DIR_ENTRADA = r".\raster_ucs"

DIR_SAIDA = r".\mosaicos_8dias"


# ============================================================
# GRADE ESPACIAL CANÔNICA
# ============================================================

X_MIN = -74.0
Y_MIN = -34.0

X_MAX = -25.0
Y_MAX = 5.0

RESOLUCAO = 0.05

CRS_DESTINO = "EPSG:4326"


# ============================================================
# DIMENSÕES FIXAS
# ============================================================

LARGURA = round(
    (X_MAX - X_MIN)
    / RESOLUCAO
)

ALTURA = round(
    (Y_MAX - Y_MIN)
    / RESOLUCAO
)


# ============================================================
# EXTENSÕES
# ============================================================

EXTENSOES = [
    "*.tif",
    "*.tiff",
]


# ============================================================
# INFORMAÇÕES DA GRADE
# ============================================================

def imprimir_grade():
    """
    Exibe as características da grade espacial
    canônica utilizada nos mosaicos.
    """

    print("=" * 80)

    print(
        "GRADE DO DATASET"
    )

    print("=" * 80)

    print(
        f"Longitude : {X_MIN} até {X_MAX}"
    )

    print(
        f"Latitude  : {Y_MIN} até {Y_MAX}"
    )

    print(
        f"Resolução : {RESOLUCAO} grau"
    )

    print(
        f"Dimensão  : {LARGURA} x {ALTURA} pixels"
    )

    print(
        f"CRS       : {CRS_DESTINO}"
    )

    print("=" * 80)

    print()


# ============================================================
# NOME DO GRUPO TEMPORAL
# ============================================================

def obter_nome_grupo(arquivo):
    """
    Remove o cd_cnuc do final do nome do arquivo.

    Exemplo:

    GC1SG1_20180202D08D_D0000_3MSG_CHLAM_3000-000123.tif

    retorna:

    GC1SG1_20180202D08D_D0000_3MSG_CHLAM_3000
    """

    nome = os.path.splitext(
        os.path.basename(
            arquivo
        )
    )[0]

    if "-" not in nome:
        return nome

    return nome.rsplit(
        "-",
        1
    )[0]


# ============================================================
# IDENTIFICAÇÃO DO ANO
# ============================================================

def obter_ano_raster(arquivo):
    """
    Obtém o ano a partir do primeiro diretório
    relativo a DIR_ENTRADA.
    """

    relativo = os.path.relpath(
        arquivo,
        DIR_ENTRADA
    )

    partes = relativo.split(
        os.sep
    )

    if len(partes) > 1:
        return partes[0]

    return "sem_ano"


# ============================================================
# LOCALIZAÇÃO DOS RASTERS
# ============================================================

def localizar_rasters():
    """
    Localiza recursivamente todos os arquivos
    GeoTIFF no diretório de entrada.
    """

    arquivos = []

    for extensao in EXTENSOES:

        arquivos.extend(

            glob.glob(
                os.path.join(
                    DIR_ENTRADA,
                    "**",
                    extensao
                ),
                recursive=True
            )

        )

    arquivos = sorted(
        set(
            arquivos
        )
    )

    print("=" * 80)

    print(
        f"Rasters encontrados: {len(arquivos)}"
    )

    print("=" * 80)

    print()

    return arquivos


# ============================================================
# AGRUPAMENTO TEMPORAL
# ============================================================

def agrupar_rasters(arquivos):
    """
    Agrupa os rasters por:

    ano + janela temporal de 8 dias.
    """

    grupos = defaultdict(
        list
    )

    for arquivo in arquivos:

        ano = obter_ano_raster(
            arquivo
        )

        nome_grupo = obter_nome_grupo(
            arquivo
        )

        chave = (
            ano,
            nome_grupo
        )

        grupos[chave].append(
            arquivo
        )

    print(
        f"Intervalos temporais encontrados: "
        f"{len(grupos)}"
    )

    print()

    return grupos


# ============================================================
# METADADOS DO RASTER DE REFERÊNCIA
# ============================================================

def obter_metadados_referencia(
    arquivo_referencia
):
    """
    Abre o primeiro raster do grupo e obtém
    os metadados radiométricos usados no mosaico.
    """

    referencia = gdal.Open(
        arquivo_referencia,
        gdal.GA_ReadOnly
    )

    if referencia is None:

        raise RuntimeError(
            "Não foi possível abrir "
            "o raster de referência."
        )

    banda = referencia.GetRasterBand(
        1
    )

    scale = banda.GetScale()

    offset = banda.GetOffset()

    nodata = banda.GetNoDataValue()

    tipo_dado = banda.DataType

    metadata_banda = banda.GetMetadata()

    # --------------------------------------------------------
    # Fallback de NoData
    # --------------------------------------------------------

    if nodata is None:

        if tipo_dado == gdal.GDT_UInt16:

            nodata = 65535

        else:

            nodata = -9999

    referencia = None

    return {
        "scale": scale,
        "offset": offset,
        "nodata": nodata,
        "tipo_dado": tipo_dado,
        "metadata_banda": metadata_banda,
    }


# ============================================================
# DIRETÓRIO / ARQUIVO DE SAÍDA
# ============================================================

def obter_arquivo_saida(
    ano,
    nome_grupo
):
    """
    Cria o diretório do ano e retorna
    o caminho do mosaico de saída.
    """

    dir_saida_ano = os.path.join(
        DIR_SAIDA,
        ano
    )

    os.makedirs(
        dir_saida_ano,
        exist_ok=True
    )

    return os.path.join(
        dir_saida_ano,
        nome_grupo + ".tif"
    )


# ============================================================
# CRIAÇÃO DO MOSAICO
# ============================================================

def criar_mosaico(
    rasters,
    arquivo_saida,
    nodata
):
    """
    Cria o mosaico diretamente com gdal.Warp
    usando uma grade espacial fixa.
    """

    warp_options = gdal.WarpOptions(

        format="GTiff",

        # ----------------------------------------------------
        # CRS
        # ----------------------------------------------------

        dstSRS=CRS_DESTINO,

        # ----------------------------------------------------
        # Extensão fixa
        # ----------------------------------------------------

        outputBounds=[
            X_MIN,
            Y_MIN,
            X_MAX,
            Y_MAX,
        ],

        # ----------------------------------------------------
        # Dimensão fixa
        # ----------------------------------------------------

        width=LARGURA,

        height=ALTURA,

        # ----------------------------------------------------
        # NoData
        # ----------------------------------------------------

        srcNodata=nodata,

        dstNodata=nodata,

        # ----------------------------------------------------
        # Reamostragem
        # ----------------------------------------------------

        resampleAlg="near",

        multithread=True,

        # ----------------------------------------------------
        # GeoTIFF
        # ----------------------------------------------------

        creationOptions=[
            "COMPRESS=DEFLATE",
            "TILED=YES",
            "BIGTIFF=IF_SAFER",
        ]
    )

    mosaico = gdal.Warp(
        arquivo_saida,
        rasters,
        options=warp_options
    )

    if mosaico is None:

        raise RuntimeError(
            "gdal.Warp retornou None."
        )

    mosaico = None


# ============================================================
# METADADOS DO MOSAICO
# ============================================================

def gravar_metadados_mosaico(
    arquivo_saida,
    rasters,
    metadados
):
    """
    Reabre o mosaico em modo de edição e
    grava metadados radiométricos e de proveniência.
    """

    dataset = gdal.Open(
        arquivo_saida,
        gdal.GA_Update
    )

    if dataset is None:

        raise RuntimeError(
            "Não foi possível reabrir o mosaico."
        )

    banda = dataset.GetRasterBand(
        1
    )

    scale = metadados["scale"]

    offset = metadados["offset"]

    nodata = metadados["nodata"]

    metadata_banda = (
        metadados["metadata_banda"]
    )

    # --------------------------------------------------------
    # Scale
    # --------------------------------------------------------

    if scale is not None:

        banda.SetScale(
            scale
        )

        banda.SetMetadataItem(
            "SCALE_FACTOR",
            str(scale)
        )

    # --------------------------------------------------------
    # Offset
    # --------------------------------------------------------

    if offset is not None:

        banda.SetOffset(
            offset
        )

        banda.SetMetadataItem(
            "ADD_OFFSET",
            str(offset)
        )

    # --------------------------------------------------------
    # NoData
    # --------------------------------------------------------

    banda.SetNoDataValue(
        nodata
    )

    # --------------------------------------------------------
    # Metadados da banda original
    # --------------------------------------------------------

    if metadata_banda:

        for chave, valor in (
            metadata_banda.items()
        ):

            banda.SetMetadataItem(
                chave,
                str(valor)
            )

    # --------------------------------------------------------
    # Metadados do dataset
    # --------------------------------------------------------

    dataset.SetMetadataItem(
        "PRODUCT",
        "SDG 14.1.1a - Marine Conservation Units Mosaic"
    )

    dataset.SetMetadataItem(
        "SOURCE",
        "JAXA GCOM-C SGLI"
    )

    dataset.SetMetadataItem(
        "TEMPORAL_COMPOSITE",
        "8 days"
    )

    dataset.SetMetadataItem(
        "NUMBER_OF_UC_RASTERS",
        str(
            len(rasters)
        )
    )

    dataset.SetMetadataItem(
        "X_MIN",
        str(X_MIN)
    )

    dataset.SetMetadataItem(
        "X_MAX",
        str(X_MAX)
    )

    dataset.SetMetadataItem(
        "Y_MIN",
        str(Y_MIN)
    )

    dataset.SetMetadataItem(
        "Y_MAX",
        str(Y_MAX)
    )

    dataset.SetMetadataItem(
        "SPATIAL_RESOLUTION",
        str(RESOLUCAO)
    )

    dataset.SetMetadataItem(
        "RASTER_WIDTH",
        str(LARGURA)
    )

    dataset.SetMetadataItem(
        "RASTER_HEIGHT",
        str(ALTURA)
    )

    dataset.SetMetadataItem(
        "CRS",
        CRS_DESTINO
    )

    dataset.SetMetadataItem(
        "PROCESSING",
        (
            "Mosaic created from individual "
            "CNUC marine conservation unit rasters "
            "on a fixed 0.05 degree grid."
        )
    )

    # --------------------------------------------------------
    # Gravação
    # --------------------------------------------------------

    banda.FlushCache()

    dataset.FlushCache()

    dataset = None


# ============================================================
# PROCESSAMENTO DE UM GRUPO
# ============================================================

def processar_grupo(
    ano,
    nome_grupo,
    rasters,
    indice,
    total_grupos
):
    """
    Processa um grupo correspondente a uma
    janela temporal de 8 dias.

    Retorna:
    "criado"
    "existente"
    "erro"
    """

    print()

    print("=" * 80)

    print(
        f"Mosaico {indice}/{total_grupos}"
    )

    print(
        f"Ano: {ano}"
    )

    print(
        f"Período: {nome_grupo}"
    )

    print(
        f"UCs/Rasters: {len(rasters)}"
    )

    print("=" * 80)

    try:

        # ----------------------------------------------------
        # 1. Metadados do primeiro raster
        # ----------------------------------------------------

        metadados = obter_metadados_referencia(
            rasters[0]
        )

        print(
            f"Scale       : "
            f"{metadados['scale']}"
        )

        print(
            f"Offset      : "
            f"{metadados['offset']}"
        )

        print(
            f"NoData      : "
            f"{metadados['nodata']}"
        )

        print(
            f"Grade saída : "
            f"{LARGURA} x {ALTURA}"
        )

        # ----------------------------------------------------
        # 2. Caminho do mosaico
        # ----------------------------------------------------

        arquivo_saida = obter_arquivo_saida(
            ano,
            nome_grupo
        )

        # ----------------------------------------------------
        # 3. Já existe?
        # ----------------------------------------------------

        if os.path.exists(
            arquivo_saida
        ):

            print(
                f"Já existe: {arquivo_saida}"
            )

            return "existente"

        # ----------------------------------------------------
        # 4. Mosaico
        # ----------------------------------------------------

        criar_mosaico(
            rasters,
            arquivo_saida,
            metadados["nodata"]
        )

        # ----------------------------------------------------
        # 5. Metadados
        # ----------------------------------------------------

        gravar_metadados_mosaico(
            arquivo_saida,
            rasters,
            metadados
        )

        print()

        print(
            f"OK: {arquivo_saida}"
        )

        return "criado"

    except Exception as erro:

        print()

        print(
            f"ERRO: {erro}"
        )

        return "erro"


# ============================================================
# PROCESSAMENTO DE TODOS OS GRUPOS
# ============================================================

def processar_grupos(grupos):
    """
    Processa todos os grupos temporais encontrados.
    """

    totais = {
        "mosaicos": 0,
        "existentes": 0,
        "erros": 0,
    }

    total_grupos = len(
        grupos
    )

    for indice, (
        (ano, nome_grupo),
        rasters
    ) in enumerate(
        sorted(
            grupos.items()
        ),
        start=1
    ):

        resultado = processar_grupo(
            ano,
            nome_grupo,
            rasters,
            indice,
            total_grupos
        )

        if resultado == "criado":

            totais["mosaicos"] += 1

        elif resultado == "existente":

            totais["existentes"] += 1

        else:

            totais["erros"] += 1

    return totais


# ============================================================
# RESUMO FINAL
# ============================================================

def imprimir_resumo(
    grupos,
    totais
):
    """
    Exibe o resumo final do processamento.
    """

    print()

    print("=" * 80)

    print(
        "PROCESSAMENTO CONCLUÍDO"
    )

    print("=" * 80)

    print(
        f"Intervalos encontrados : "
        f"{len(grupos)}"
    )

    print(
        f"Mosaicos criados       : "
        f"{totais['mosaicos']}"
    )

    print(
        f"Já existentes          : "
        f"{totais['existentes']}"
    )

    print(
        f"Erros                  : "
        f"{totais['erros']}"
    )

    print()

    print(
        f"Diretório de saída: "
        f"{DIR_SAIDA}"
    )

    print("=" * 80)


# ============================================================
# MAIN
# ============================================================

def main():
    """
    Função principal do programa.
    """

    # --------------------------------------------------------
    # 1. Prepara diretório
    # --------------------------------------------------------

    os.makedirs(
        DIR_SAIDA,
        exist_ok=True
    )

    # --------------------------------------------------------
    # 2. Exibe grade
    # --------------------------------------------------------

    imprimir_grade()

    # --------------------------------------------------------
    # 3. Localiza rasters
    # --------------------------------------------------------

    arquivos = localizar_rasters()

    if not arquivos:

        print(
            "Nenhum raster encontrado."
        )

        return

    # --------------------------------------------------------
    # 4. Agrupa por janela de 8 dias
    # --------------------------------------------------------

    grupos = agrupar_rasters(
        arquivos
    )

    # --------------------------------------------------------
    # 5. Processa mosaicos
    # --------------------------------------------------------

    totais = processar_grupos(
        grupos
    )

    # --------------------------------------------------------
    # 6. Resumo
    # --------------------------------------------------------

    imprimir_resumo(
        grupos,
        totais
    )


# ============================================================
# PONTO DE ENTRADA
# ============================================================

if __name__ == "__main__":
    main()


## 2.8 Extrair as variáveis das imagens GeoTiff -> $y = w(x) + b$

O NetCDF contém os seguintes "meta-dados" que serão muito cruciais ao geoprocessamento da imagem:

SCALE: é o valor do peso `w`
OFFSET: é o valor do viés `b`

- CHLA_AVE: média da concentração de clorofila-a;
- CHLA_QA_flag: informação de qualidade da observação;
- CHLA_STD: desvio padrão associado à concentração de clorofila-a.

Vamos usar a ferramenta GdalInfo da biblioteca GDAL para extair os valores de `w` e `b` propostos no sensor.

In [ ]:
# Usando o netcdf procure pelos metadados w (scale_factor) e b (add_offset)
!gdalinfo NETCDF:GC1SG1_20180101A08D_D0000_3MSG_CHLAM_3000.nc:CHLA_AVE

# Usando o geotiff procure pelos metadados w (scale_factor) e b (add_offset)
!gdalinfo GC1SG1_20180101A08D_D0000_3MSG_CHLAM_3000.tif

In [ ]:
import os
import csv
import glob
import numpy as np
from osgeo import gdal, ogr

gdal.UseExceptions()

# ============================================================
# CONFIGURAÇÃO
# ============================================================

# Camada vetorial contendo as UCs.
# Pode ser a camada já reprojetada para EPSG:4326.
ARQUIVO_UCS = r".\camada_vetorial_cnuc\ucs4326.shp"

# Diretório onde estão:
#
# raster_ucs/
# ├── 2018/
# ├── 2019/
# ...
# └── 2026/
#
DIR_RASTERS = r".\raster_ucs"

# CSV final
ARQUIVO_CSV = r".\data\ods1411a_estatisticas_ucs.csv"

# Campo que relaciona raster e UC
CAMPO_CHAVE = "cd_cnuc"

# Cria diretório de saída
os.makedirs(
    os.path.dirname(ARQUIVO_CSV),
    exist_ok=True
)


# ============================================================
# FUNÇÃO PARA NORMALIZAR cd_cnuc
# ============================================================

def normalizar_codigo(valor):
    """
    Normaliza o código CNUC para comparação.

    Mantém o código como texto para não perder zeros
    à esquerda.
    """

    if valor is None:
        return None

    valor = str(valor).strip()

    if valor == "":
        return None

    return valor


# ============================================================
# 1. ABRE A CAMADA VETORIAL
# ============================================================

print("=" * 80)
print("Lendo camada das Unidades de Conservação...")
print("=" * 80)

ds_vector = ogr.Open(
    ARQUIVO_UCS,
    0
)

if ds_vector is None:
    raise RuntimeError(
        f"Não foi possível abrir: {ARQUIVO_UCS}"
    )

layer = ds_vector.GetLayer()


# ============================================================
# 2. IDENTIFICA TODOS OS CAMPOS DO SHAPEFILE
# ============================================================

layer_defn = layer.GetLayerDefn()

campos_uc = []

for i in range(
    layer_defn.GetFieldCount()
):

    campo = layer_defn.GetFieldDefn(i)

    campos_uc.append(
        campo.GetName()
    )


print(
    f"Campos encontrados: {len(campos_uc)}"
)

print(
    ", ".join(campos_uc)
)


# ============================================================
# 3. CARREGA ATRIBUTOS DAS UCs EM MEMÓRIA
# ============================================================

ucs = {}

layer.ResetReading()

total_marinho = 0


for feature in layer:

    # --------------------------------------------------------
    # Mantém apenas UCs marinhas
    # --------------------------------------------------------

    marinho = feature.GetField(
        "marinho"
    )

    if marinho is None:
        continue

    if str(marinho).strip() == "":
        continue


    # --------------------------------------------------------
    # Obtém cd_cnuc
    # --------------------------------------------------------

    codigo = normalizar_codigo(
        feature.GetField(
            CAMPO_CHAVE
        )
    )

    if codigo is None:
        continue


    # --------------------------------------------------------
    # Copia TODOS os atributos
    # --------------------------------------------------------

    atributos = {}

    for campo in campos_uc:

        atributos[campo] = (
            feature.GetField(campo)
        )


    ucs[codigo] = atributos

    total_marinho += 1


print()
print(
    f"UCs marinhas carregadas: {total_marinho}"
)

print()


# ============================================================
# 4. LOCALIZA TODOS OS GEOTIFFS
# ============================================================

arquivos_raster = glob.glob(
    os.path.join(
        DIR_RASTERS,
        "**",
        "*.tif"
    ),
    recursive=True
)

arquivos_raster += glob.glob(
    os.path.join(
        DIR_RASTERS,
        "**",
        "*.tiff"
    ),
    recursive=True
)

arquivos_raster = sorted(
    set(arquivos_raster)
)


print("=" * 80)
print(
    f"Rasters encontrados: {len(arquivos_raster)}"
)
print("=" * 80)
print()


# ============================================================
# 5. CAMPOS ADICIONAIS DO CSV
# ============================================================

campos_raster = [

    # Identificação
    "raster",
    "ano",

    # Metadados radiométricos
    "scale",
    "offset",
    "nodata",

    # Quantidade de pixels
    "n_pixels_validos",

    # Estatísticas já convertidas para valores físicos
    "mediana",
    "media",
    "minimo",
    "maximo"
]


# ============================================================
# 6. CABEÇALHO FINAL
# ============================================================

cabecalho = (
    campos_uc
    + campos_raster
)


# ============================================================
# 7. ABRE CSV
# ============================================================

# utf-8-sig facilita abertura no Excel no Windows.
#
# delimiter=";" é conveniente em instalações brasileiras
# do Excel.

with open(
    ARQUIVO_CSV,
    "w",
    newline="",
    encoding="utf-8-sig"
) as csvfile:

    writer = csv.DictWriter(
        csvfile,
        fieldnames=cabecalho,
        delimiter=";"
    )

    writer.writeheader()


    # ========================================================
    # 8. PROCESSA CADA RASTER
    # ========================================================

    total_ok = 0
    total_sem_uc = 0
    total_sem_pixels = 0
    total_erros = 0


    for indice, arquivo_raster in enumerate(
        arquivos_raster,
        start=1
    ):

        print(
            f"[{indice}/{len(arquivos_raster)}] "
            f"{arquivo_raster}"
        )


        try:

            # =================================================
            # 9. ABRE O RASTER
            # =================================================

            dataset = gdal.Open(
                arquivo_raster,
                gdal.GA_ReadOnly
            )

            if dataset is None:

                print(
                    "  ERRO: não foi possível abrir."
                )

                total_erros += 1

                continue


            banda = dataset.GetRasterBand(1)


            # =================================================
            # 10. LÊ METADADOS
            # =================================================

            scale = banda.GetScale()

            offset = banda.GetOffset()

            nodata = banda.GetNoDataValue()


            # -------------------------------------------------
            # Valores utilizados nos cálculos
            # -------------------------------------------------

            scale_calculo = (
                scale
                if scale is not None
                else 1.0
            )

            offset_calculo = (
                offset
                if offset is not None
                else 0.0
            )


            # =================================================
            # 11. IDENTIFICA A UC
            # =================================================

            # Primeiro tenta usar o próprio metadado
            # gravado anteriormente no GeoTIFF.

            codigo = dataset.GetMetadataItem(
                "CNUC_CODE"
            )

            codigo = normalizar_codigo(
                codigo
            )


            # -------------------------------------------------
            # Se o raster não tiver CNUC_CODE nos metadados,
            # extrai do nome:
            #
            # raster-original-cd_cnuc.tif
            # -------------------------------------------------

            if codigo is None:

                nome_sem_extensao = (
                    os.path.splitext(
                        os.path.basename(
                            arquivo_raster
                        )
                    )[0]
                )

                if "-" in nome_sem_extensao:

                    codigo = (
                        nome_sem_extensao
                        .rsplit("-", 1)[1]
                    )

                    codigo = normalizar_codigo(
                        codigo
                    )


            # =================================================
            # 12. PROCURA A UC NO SHAPEFILE
            # =================================================

            if codigo not in ucs:

                print(
                    f"  AVISO: cd_cnuc não encontrado: "
                    f"{codigo}"
                )

                total_sem_uc += 1

                dataset = None

                continue


            atributos_uc = ucs[codigo]


            # =================================================
            # 13. LÊ OS PIXELS
            # =================================================

            dados = banda.ReadAsArray()

            if dados is None:

                print(
                    "  ERRO: não foi possível ler pixels."
                )

                total_erros += 1

                dataset = None

                continue


            # Converte para float64 para evitar problemas
            # durante os cálculos.
            dados = dados.astype(
                np.float64
            )


            # =================================================
            # 14. CRIA MÁSCARA DE PIXELS VÁLIDOS
            # =================================================

            mascara = np.isfinite(
                dados
            )


            # -------------------------------------------------
            # Elimina NoData
            # -------------------------------------------------

            if nodata is not None:

                if np.isnan(nodata):

                    mascara &= ~np.isnan(
                        dados
                    )

                else:

                    mascara &= (
                        dados != nodata
                    )


            # =================================================
            # 15. SELECIONA PIXELS VÁLIDOS
            # =================================================

            pixels_dn = dados[
                mascara
            ]


            if pixels_dn.size == 0:

                print(
                    "  AVISO: nenhum pixel válido."
                )

                total_sem_pixels += 1

                dataset = None

                continue


            # =================================================
            # 16. APLICA SCALE E OFFSET
            # =================================================

            pixels = (
                pixels_dn
                * scale_calculo
                + offset_calculo
            )


            # =================================================
            # 17. CALCULA ESTATÍSTICAS
            # =================================================

            n_pixels = int(
                pixels.size
            )

            mediana = float(
                np.median(pixels)
            )

            media = float(
                np.mean(pixels)
            )

            minimo = float(
                np.min(pixels)
            )

            maximo = float(
                np.max(pixels)
            )


            # =================================================
            # 18. IDENTIFICA O ANO
            # =================================================

            relativo = os.path.relpath(
                arquivo_raster,
                DIR_RASTERS
            )

            partes = relativo.split(
                os.sep
            )

            if len(partes) > 1:
                ano = partes[0]
            else:
                ano = ""


            # =================================================
            # 19. MONTA LINHA DO CSV
            # =================================================

            linha = {}

            # -------------------------------------------------
            # Todos os atributos do shapefile
            # -------------------------------------------------

            for campo in campos_uc:

                linha[campo] = (
                    atributos_uc.get(
                        campo
                    )
                )


            # -------------------------------------------------
            # Informações do raster
            # -------------------------------------------------

            linha["raster"] = (
                os.path.basename(
                    arquivo_raster
                )
            )

            linha["ano"] = ano

            linha["scale"] = scale

            linha["offset"] = offset

            linha["nodata"] = nodata

            linha["n_pixels_validos"] = (
                n_pixels
            )

            linha["mediana"] = mediana

            linha["media"] = media

            linha["minimo"] = minimo

            linha["maximo"] = maximo


            # =================================================
            # 20. GRAVA LINHA
            # =================================================

            writer.writerow(
                linha
            )


            total_ok += 1


            print(
                f"  cd_cnuc : {codigo}"
            )

            print(
                f"  pixels  : {n_pixels}"
            )

            print(
                f"  mediana : {mediana}"
            )

            print(
                f"  média   : {media}"
            )

            print(
                f"  mínimo  : {minimo}"
            )

            print(
                f"  máximo  : {maximo}"
            )


            dataset = None


        except Exception as erro:

            total_erros += 1

            print(
                f"  ERRO: {erro}"
            )


# ============================================================
# 21. RESULTADO FINAL
# ============================================================

ds_vector = None

print()
print("=" * 80)
print("PROCESSAMENTO CONCLUÍDO")
print("=" * 80)

print(
    f"Rasters processados com sucesso : {total_ok}"
)

print(
    f"Rasters sem UC correspondente   : {total_sem_uc}"
)

print(
    f"Rasters sem pixels válidos      : {total_sem_pixels}"
)

print(
    f"Erros                           : {total_erros}"
)

print()
print(
    f"CSV gerado:"
)

print(
    ARQUIVO_CSV
)

print("=" * 80)

## 2.9 - Formatação temporal: acrescentar data de início e data de fim no dataset

In [ ]:
import pandas as pd

# ============================================================
# Arquivos
# ============================================================

arquivo_entrada = r".\data\ods1411a_estatisticas_ucs.csv"
arquivo_saida = r".\data\ods1411a_estatisticas_ucs_formatado.csv"


# ============================================================
# 1. Lê o CSV
# ============================================================

df = pd.read_csv(
    arquivo_entrada,
    sep=";",
    encoding="utf-8-sig"
)


# ============================================================
# 2. Extrai YYYYMMDD do nome do raster
#
# Exemplo:
#
# GC1SG1_20180101D08D_D0000_3MSG_CHLAM_3000-0000_00_0001.tif
#        ^^^^^^^^
# ============================================================

df["data_inicio"] = df["raster"].str.extract(
    r"GC1SG1_(\d{8})"
)[0]


# ============================================================
# 3. Converte para data
# ============================================================

df["data_inicio"] = pd.to_datetime(
    df["data_inicio"],
    format="%Y%m%d",
    errors="coerce"
)


# ============================================================
# 4. Calcula o último dia da janela de 8 dias
#
# Dia inicial + 7 dias = 8 dias contando o primeiro.
# ============================================================

df["data_fim"] = (
    df["data_inicio"]
    + pd.Timedelta(days=7)
)


# ============================================================
# 5. Converte para ISO 8601
#
# 2018-01-01
# ============================================================

df["data_inicio"] = df["data_inicio"].dt.strftime(
    "%Y-%m-%d"
)

df["data_fim"] = df["data_fim"].dt.strftime(
    "%Y-%m-%d"
)


# ============================================================
# 6. Salva
# ============================================================

df.to_csv(
    arquivo_saida,
    sep=";",
    index=False,
    encoding="utf-8-sig"
)

print(f"CSV gerado: {arquivo_saida}")

# 3. Preparação dos Dados




## 3.1 Extrair os municípios e suas populações


## 3.2 Construir o DataFrame


## 3.3 Selecionar 100 municípios



## 3.4 Verificar tipos e dados ausentes


## 3.5 Estatística descritiva


# 4. Modelagem


## 4.1 Definir X e y


## 4.2 Separar treino e teste

Vamos utilizar:

- 70% dos municípios para treino;
- 30% para teste.

O argumento `stratify=y` preserva aproximadamente a proporção das três classes nos dois conjuntos.


## 4.4 Treinar o modelo


# 5. Avaliação

Agora vamos utilizar os 30 municípios reservados para teste.

Esses municípios não foram utilizados pelo modelo durante o treinamento.


## 5.1 Comparar valores reais e previstos


## 5.2 Calcular a acurácia


## 5.3 Matriz de confusão


# 6. Implantação

Vamos simular uma pequena função que recebe a população urbana de uma nova cidade e devolve a classe prevista pelo modelo.


## 6.1 Teste com diferentes populações

Experimente diferentes valores e observe a classificação retornada pelo modelo.


# 7. Fechamento — CRISP-DM

Neste notebook percorremos o ciclo completo:

### 1. Compreensão do Negócio
Definimos o problema de classificação dos municípios.

### 2. Compreensão dos Dados
Investigamos a API REST do IBGE e a estrutura JSON.

### 3. Preparação dos Dados
Transformamos o JSON em `DataFrame`, selecionamos 100 municípios e criamos as classes.

### 4. Modelagem
Criamos e treinamos um modelo KNN.

### 5. Avaliação
Utilizamos dados de teste, calculamos acurácia e matriz de confusão.

### 6. Implantação
Criamos uma função para classificar uma nova observação.

O ciclo pode ser repetido posteriormente utilizando outras variáveis, outros estados, novos critérios de classificação ou outros algoritmos.
